# UPR-MVS: 多视图立体视觉训练框架

这是一个完整的 Jupyter Notebook 版本，整合了 UPR-MVS 项目的所有核心代码。

## 目录
1. [环境配置与导入](#环境配置与导入)
2. [工具函数](#工具函数)
3. [数据集加载](#数据集加载)
4. [模型组件](#模型组件)
5. [完整模型](#完整模型)
6. [损失函数](#损失函数)
7. [训练引擎](#训练引擎)
8. [使用示例](#使用示例)

## 环境配置与导入

In [ ]:
# 添加项目根目录到路径
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

# 标准库导入
from contextlib import nullcontext
from dataclasses import dataclass, field
import inspect
import os
import time
import warnings
from typing import Any, Callable, Iterable, Sequence, Tuple, Dict, List, Optional
from pathlib import Path as PathLib

# 第三方库导入
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.utils.checkpoint import checkpoint

# 分布式训练支持
try:
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
except ImportError:
    dist = None
    DDP = None

# TensorBoard 支持（可选）
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    SummaryWriter = None

# YAML 配置支持
try:
    import yaml
except ImportError:
    yaml = None

# 进度条支持（可选）
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 工具函数

In [ ]:
# ==================== 分布式训练工具 ====================

@dataclass(frozen=True)
class DistributedConfig:
    distributed: bool
    rank: int
    world_size: int
    local_rank: int
    device: torch.device

def is_dist_available_and_initialized() -> bool:
    return dist is not None and dist.is_available() and dist.is_initialized()

def get_rank() -> int:
    return dist.get_rank() if is_dist_available_and_initialized() else 0

def get_world_size() -> int:
    return dist.get_world_size() if is_dist_available_and_initialized() else 1

def is_main_process() -> bool:
    return get_rank() == 0

def synchronize() -> None:
    if is_dist_available_and_initialized():
        dist.barrier()

def cleanup_distributed() -> None:
    if is_dist_available_and_initialized():
        dist.destroy_process_group()

def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if hasattr(model, 'module') else model

def reduce_dict(input_dict: Dict[str, Tensor], average: bool = True) -> Dict[str, Tensor]:
    if not is_dist_available_and_initialized():
        return input_dict
    
    with torch.no_grad():
        keys = sorted(input_dict.keys())
        values = torch.stack([input_dict[key] for key in keys], dim=0)
        dist.all_reduce(values)
        if average:
            values /= float(get_world_size())
        return {key: value for key, value in zip(keys, values)}

def move_to_device(batch: Any, device: torch.device) -> Any:
    if isinstance(batch, Tensor):
        return batch.to(device, non_blocking=True)
    if isinstance(batch, dict):
        return {key: move_to_device(value, device) for key, value in batch.items()}
    if isinstance(batch, list):
        return [move_to_device(value, device) for value in batch]
    if isinstance(batch, tuple):
        return tuple(move_to_device(value, device) for value in batch)
    return batch

def init_distributed_mode(launcher: str = 'none', backend: str = 'nccl') -> DistributedConfig:
    launcher = launcher.lower()
    if launcher not in {'none', 'pytorch'}:
        raise ValueError(f'Unsupported launcher: {launcher}')
    
    world_size = int(os.environ.get('WORLD_SIZE', '1'))
    rank = int(os.environ.get('RANK', '0'))
    local_rank = int(os.environ.get('LOCAL_RANK', '0'))
    distributed = launcher == 'pytorch' and world_size > 1
    
    if torch.cuda.is_available():
        device = torch.device('cuda', local_rank if distributed else 0)
        torch.cuda.set_device(device)
    else:
        device = torch.device('cpu')
        backend = 'gloo'
    
    if distributed and not is_dist_available_and_initialized():
        if dist is not None:
            dist.init_process_group(backend=backend, init_method='env://')
            if device.type == 'cuda':
                dist.barrier(device_ids=[device.index])
            else:
                dist.barrier()
    
    return DistributedConfig(
        distributed=distributed,
        rank=rank,
        world_size=world_size,
        local_rank=local_rank,
        device=device,
    )

def save_on_master(payload: Any, path: PathLib) -> None:
    if is_main_process():
        torch.save(payload, path)

# ==================== 指标计算工具 ====================

def tensor_dict_to_floats(metrics: Dict[str, Tensor]) -> Dict[str, float]:
    return {key: float(value.detach().cpu().item()) for key, value in metrics.items()}

@dataclass
class ScalarMeter:
    sums: Dict[str, float] = field(default_factory=dict)
    count: int = 0
    
    def update(self, metrics: Dict[str, float]) -> None:
        self.count += 1
        for key, value in metrics.items():
            self.sums[key] = self.sums.get(key, 0.0) + float(value)
    
    def averages(self) -> Dict[str, float]:
        if self.count == 0:
            return {key: 0.0 for key in self.sums}
        return {key: value / self.count for key, value in self.sums.items()}

def format_metrics(metrics: Dict[str, float], keys: List[str] = None) -> str:
    if keys is None:
        keys = sorted(metrics.keys())
    return ' '.join(f'{key}={metrics[key]:.4f}' for key in keys if key in metrics)

# ==================== 检查点管理 ====================

def load_checkpoint(
    checkpoint_path: str | PathLib,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    scheduler: Any | None,
    scaler: Any | None,
    device: torch.device,
    load_training_state: bool = True,
) -> Tuple[int, float]:
    if not checkpoint_path:
        return 0, float('inf')
    
    path = PathLib(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(f'Resume checkpoint not found: {path}')
    
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    unwrap_model(model).load_state_dict(checkpoint['model'], strict=True)
    
    if load_training_state and optimizer is not None and 'optimizer' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer'])
    if load_training_state and scheduler is not None and 'scheduler' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler'])
    if load_training_state and scaler is not None and 'scaler' in checkpoint and checkpoint['scaler'] is not None:
        scaler.load_state_dict(checkpoint['scaler'])
    
    start_epoch = int(checkpoint['epoch']) + 1 if load_training_state else 0
    best_metric = float(checkpoint.get('best_metric', checkpoint.get('best_depth_abs_error', float('inf'))))
    return start_epoch, best_metric

def save_checkpoint(
    work_dir: PathLib,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    scaler: Any,
    best_metric: float,
    monitor_key: str,
    tag: str,
) -> None:
    payload = {
        'epoch': epoch,
        'model': unwrap_model(model).state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict() if scaler is not None else None,
        'best_metric': best_metric,
        'monitor_key': monitor_key,
    }
    save_on_master(payload, work_dir / f'{tag}.pth')